In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("UdemyCoursesDataset.csv")
df.head()

,course_id,course_title,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject
0,288942,#1 Piano Hand Coordination: Play 10th Ballad i...,True,35,3137,18,68,All Levels,1.5 hours,2014-09-18T05:07:05Z,Musical Instruments
1,1170074,#10 Hand Coordination - Transfer Chord Ballad ...,True,75,1593,1,41,Intermediate Level,1 hour,2017-04-12T19:06:34Z,Musical Instruments
2,1193886,#12 Hand Coordination: Let your Hands dance wi...,True,75,482,1,47,Intermediate Level,1.5 hours,2017-04-26T18:34:57Z,Musical Instruments
3,1116700,#4 Piano Hand Coordination: Fun Piano Runs in ...,True,75,850,3,43,Intermediate Level,1 hour,2017-02-21T23:48:18Z,Musical Instruments
4,1120410,#5 Piano Hand Coordination: Piano Runs in 2 ...,True,75,940,3,32,Intermediate Level,37 mins,2017-02-21T23:44:49Z,Musical Instruments


In [16]:
df.info()

# Audit Findings & Action Items:
# 1. 'price' is stored as object/str -> Needs clean-up & cast to float/int. #Done
# 2. 'content_duration' is stored as object/str -> Needs number extraction (hours). 
# 3. 'published_timestamp' is stored as object/str -> Needs conversion to datetime.

<class 'pandas.DataFrame'>
RangeIndex: 3682 entries, 0 to 3681
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   course_id            3682 non-null   int64
 1   course_title         3682 non-null   str  
 2   is_paid              3682 non-null   bool 
 3   price                3682 non-null   str  
 4   num_subscribers      3682 non-null   int64
 5   num_reviews          3682 non-null   int64
 6   num_lectures         3682 non-null   int64
 7   level                3682 non-null   str  
 8   content_duration     3682 non-null   str  
 9   published_timestamp  3682 non-null   str  
 10  subject              3682 non-null   str  
dtypes: bool(1), int64(4), str(6)
memory usage: 666.1 KB


In [17]:
print("================================================")
print("            Data Quality Audit Report           ")
print("================================================")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print(f"Duplicate Rows Count: {df.duplicated().sum()}")
print("================================================")
print("Missing Values Breakdown: ")
print(df.isnull().sum()[df.isnull().sum()>0] if df.isnull().sum().sum()>0 else "No Missing Values Found")
print("================================================")

            Data Quality Audit Report           
Total Rows: 3682
Total Columns: 11
Duplicate Rows Count: 6
Missing Values Breakdown: 
No Missing Values Found


In [18]:
df = df.drop_duplicates()

print("==========================================")
print("      DEDUPLICATION VERIFICATION          ")
print("==========================================")
print(f"Updated Rows: {df.shape[0]}")
print(f"Removed Rows Count: {3682-df.shape[0]}")
print("==========================================")

      DEDUPLICATION VERIFICATION          
Updated Rows: 3676
Removed Rows Count: 6


In [19]:
print(f"Price Unique Values: {df["price"].unique()}")

# Need to replace "Free" to "0" and dtype to Float

Price Unique Values: <ArrowStringArray>
[  '35',   '75',   '65',  '200',   '25',  '100',   '20',   '40',   '30',
   '45',  '185',  '120',  '105',   '50', 'Free',  '145',   '55',   '85',
  '110',  '150',   '95',  '180',  '125',   '60',   '90',  '195',   '80',
   '70',  '140',  '190',  '115',  '130',  '170',  '175',  '165',  '135',
  '155',  '160']
Length: 38, dtype: str


In [20]:
df['price'] = df['price'].replace('Free',"0").astype(float)
print(f"Update Price DType : {df['price'].dtype}")

Update Price DType : float64


In [21]:
df['content_duration'].unique()


<ArrowStringArray>
[   '1.5 hours',       '1 hour',      '37 mins',      '44 mins',
    '5.5 hours',      '4 hours',      '31 mins',    '2.5 hours',
    '6.5 hours',      '3 hours',
 ...
 '18 questions', '82 questions',   '78.5 hours',   '21.5 hours',
     '51 hours',   '76.5 hours',     '42 hours',     '43 hours',
     '70 hours',      '28 mins']
Length: 109, dtype: str

In [33]:
import re

def clean_duration(value): 
    if pd.isna(value): 
        return 0.0
    val_str = str(value).lower()

    if 'question' in val_str: 
        return 0.0

    match = re.search(r'([\d.]+)' , val_str)

    if not match: 
        return 0.0

    num = float(match.group(1))

    if 'min' in val_str: 
        return round(num/60 , 2)

    return num

df['content_duration'] = df['content_duration'].apply(clean_duration)


print(f"Content Duration Unique Values: \n {df['content_duration'].unique()}")
print('==================================================================')
print(f"Content Duration DType : {df['content_duration'].dtype}")
    


Content Duration Unique Values: 
 [ 1.5   1.    0.62  0.73  5.5   4.    0.52  2.5   6.5   3.    8.5   4.5
  9.5   2.    3.5   9.    5.    0.6  16.    0.53 14.    0.58  6.    0.68
  7.5  14.5  24.5   0.55 29.   13.5   0.7  33.   12.5   8.    7.    0.57
 11.5  62.   12.    0.65  0.32 15.5  10.5  21.   68.5  19.5  20.5  22.
 10.    0.63  0.72  0.67 24.   13.   31.5  44.5  17.5  48.5  60.   11.
 27.5  18.5  25.5  22.5  16.5   0.5  57.   45.   20.   39.   34.   25.
 26.   29.5  15.   37.5  71.5  47.   18.   38.   28.5   0.13 32.5  31.
 17.    0.45  0.48 46.5  43.5  66.5  26.5   0.   30.   30.5  19.   23.
 23.5  78.5  21.5  51.   76.5  42.   43.   70.    0.47]
Content Duration DType : float64
